# Psalm 42 — Full 20:00 MasterSync • Kaggle GPU Build

**Why Kaggle is better than the current 3.8GB sandbox:**
- Sandbox: no GPU, 3.8 GB RAM → monolithic 45-input `xfade` OOM (364 MB truncated, no `moov` → unrecoverable)
- Kaggle: `T4×2 (16 GB VRAM)`, `>13 GB RAM`, `9h` runtime, `20 GB` persistent disk → can run **Wan2.2-I2V 14B + LongCat-Video 13.6B + LeviTor + AnyI2V** for real motion, not just pan/zoom.

**This notebook:** clones your 8 repos, downloads weights, runs the *chunked* resume-safe build (also works on sandbox), and optionally runs *real AI motion* via ComfyUI.

1. Set `HF_TOKEN` in Kaggle Secrets (for gated models) if needed
2. Run All — output `/kaggle/working/FULL-20min-PSALM42-MasterSync.mp4` auto-uploaded to Kaggle Output


## 0 — Environment check


In [ ]:
!nvidia-smi
!free -h && df -h /kaggle/working
!python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"


## 1 — Clone your repos (with resume)


In [ ]:
import os, subprocess, pathlib
WORK = pathlib.Path("/kaggle/working/youtube-videos")
if WORK.exists():
    print("Updating existing clone...")
    subprocess.check_call(["git","-C",str(WORK),"fetch","origin"])
    subprocess.check_call(["git","-C",str(WORK),"reset","--hard","origin/arena/d58bab72-youtube-videos"])
    subprocess.check_call(["git","-C",str(WORK),"clean","-fd"])
else:
    subprocess.check_call(["git","clone","https://github.com/wambetebenjamin/youtube-videos.git", str(WORK)])
    subprocess.check_call(["git","-C",str(WORK),"checkout","arena/d58bab72-youtube-videos"])
os.chdir(WORK)
import subprocess as _sp
print(f"Branch: {_sp.check_output(['git','-C',str(WORK),'rev-parse','--abbrev-ref','HEAD']).decode().strip()}")
print(f"Commit: {_sp.check_output(['git','-C',str(WORK),'rev-parse','--short','HEAD']).decode().strip()}")
!ls -lh docs/master-20min-ink-doodle/audio-full/ 2>&1 | head -20
!ls docs/master-20min-ink-doodle/audio-full/*.mp3 2>&1 | head -20


## 2 — (Optional) Download real AI weights — only on Kaggle GPU
Run this if you want *true* Wan2.2 / AnyI2V / LeviTor motion. Skippable — the chunked FFmpeg build works without it.


In [ ]:
# !pip install -q huggingface_hub
# from huggingface_hub import hf_hub_download
# # Wan2.2-I2V-A14B
# # hf_hub_download(repo_id="Wan-AI/Wan2.2-I2V-A14B", filename="...", local_dir="/kaggle/working/weights/wan22")
# print("Weights stub — uncomment when you have HF_TOKEN and accept licenses")
print("Skipped — FFmpeg pan/zoom already gives seamless 20:00. Enable above for real AI motion.")


## 3 — Resume-safe chunked build (the fix for your 364 MB failure)

The old single `ffmpeg -i 45 images -filter_complex xfade` needed >5 GB RAM → killed mid-mux → no `moov` → **cannot continue**. 
This chunked version saves each segment individually. If killed at seg 18, restarting skips 0–18.


In [ ]:
import os
os.chdir("/kaggle/working/youtube-videos")
!ls docs/master-20min-ink-doodle/audio-full/*.mp3
# Build voiceover 11.317m -> padded 20:00 (same as sandbox)
!python -c "import imageio_ffmpeg; print(imageio_ffmpeg.get_ffmpeg_exe())"
import subprocess
FF=subprocess.check_output(["python3","-c","import imageio_ffmpeg; print(imageio_ffmpeg.get_ffmpeg_exe())"]).decode().strip()
# If full-voice-1200.m4a not yet present, build it
import pathlib
voice=pathlib.Path("/kaggle/working/youtube-videos/full-voice-1200.m4a")
if not voice.exists():
    # concat 9 mp3s + apad to 1200s
    ins=[]
    for p in sorted(pathlib.Path("docs/master-20min-ink-doodle/audio-full").glob("*.mp3")):
        ins.extend(["-i", str(p)])
    filt=" ".join([f"[{i}:a]" for i in range(9)])+f"concat=n=9:v=0:a=1,apad,atrim=duration=1200[a]"
    cmd=[FF,"-y"]+ins+["-filter_complex",filt,"-map","[a]","-c:a","aac","-b:a","128k",str(voice)]
    subprocess.check_call(cmd)
    print(f"voice {voice.stat().st_size}")
else:
    print(f"voice exists {voice.stat().st_size}")


In [ ]:
# Run the chunked video builder (resumes per segment)
!python /kaggle/working/youtube-videos/kaggle/build_chunked.py 2>&1 | tail -n 100
# This writes:
#  /kaggle/working/youtube-videos/tmp_segments/seg00..seg44.mp4
#  /kaggle/working/youtube-videos/chunk00..chunk05.mp4
#  /kaggle/working/FULL-20min-PSALM42-MasterSync.mp4
!ls -lh /kaggle/working/*.mp4 /kaggle/working/youtube-videos/chunk*.mp4 /kaggle/working/youtube-videos/tmp_segments/*.mp4 2>&1 | head -20


## 4 — (Optional) Real AI motion via ComfyUI workflows

If you downloaded weights in §2, you can now run the 3 routed workflows (see `out/pipeline-sample/workflows/*.json`). Each scene type uses a different model — this is what gives you true flicker / stacking / pullback instead of simulated zoom.

```
TYPE A Wan2.2-I2V  → candle, rain, ruins    (ambient)
TYPE B AnyI2V      → boxes, rising, chains  (character)
TYPE C LeviTor     → candle pullback, well  (camera)
LongCat-Video      → extend 4s → 45s without looping
```


In [ ]:
# Stub: integrate ComfyUI custom nodes once weights present
# !git clone https://github.com/comfyanonymous/ComfyUI /kaggle/working/ComfyUI
# !cp -r image-to-video-repos/Wan2.2 /kaggle/working/ComfyUI/custom_nodes/ ...
print("Enable when GPU weights ready — workflows in out/pipeline-sample/workflows/ are valid ComfyUI JSON")


## 5 — Upload result


In [ ]:
import pathlib, shutil, os
src=pathlib.Path("/kaggle/working/FULL-20min-PSALM42-MasterSync.mp4")
dst=pathlib.Path("/kaggle/working/youtube-videos/FULL-20min-PSALM42-MasterSync.mp4")
if src.exists():
    shutil.copy(src, dst)
    print(f"Ready for Kaggle Output: {dst} {dst.stat().st_size/1e6:.1f} MB")
    # Also copy chunks for GitHub HTML player
    for c in pathlib.Path("/kaggle/working/youtube-videos").glob("chunk*.mp4"):
        print(c, c.stat().st_size)
else:
    print("Not yet — build still running, check tail above")
